In [2]:
# !pip install --upgrade openai
# !pip install pinecone
# !pip install langchain
# !pip install langchain-community
# !pip install pandas
# !pip install dotenv
# !pip install numpy

# !pip install pydantic

In [3]:
from openai import OpenAI
import os
from pinecone import Pinecone, ServerlessSpec
from datetime import datetime

from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Pinecone as LangchainPinecone
from langchain.vectorstores import Chroma
from langchain.schema import Document

import pandas as pd 
import numpy as np

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
import json

In [4]:
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY') 
# os.environ["PINECONE_API_KEY"] = os.getenv('PINECONE_API_KEY')


In [5]:
client = OpenAI()
# date= datetime.now().strftime("%Y-%m-%d")

prompt = '''

Goal:
I want a list of the best News websites which shows the top engaging news from whole world. While returning, give 5 websites which we have given below and 5 websites from your side. Remember, the 5 websites that you will give needs to be different from those that we have provided you.

Our websites -

international-
https://www.theguardian.com/international
https://www.dw.com/en
https://www.cnn.com

indian-
https://indianexpress.com

china-
https://www.scmp.com


Return Format:
We will provide 5 websites and you have to add 5 more websites to the list .out of which 3 should be sites focused on global top news ,1 should be focused on news from india and 1 should be focused on china. In total, give me 10 websites


Warning:
Be careful to make sure that the name and website address is correct .and the websites are trustworthy to show true and unobstructed news.
as given above 5 sites should be included in the list and should follow the EXACT format as shown in example. 
Make sure in the end we have exactly 10 sites in the Final list. NO MORE NO LESS. 

Example: News Website:
international-
1.https://www.theguardian.com/international
2.https://www.dw.com/en
3.https://www.cnn.com
4.[Your site 1]
5.[Your site 2]
6.[Your site 3]

indian-
7.https://indianexpress.com
8.[Your site 4]


china-
9.https://www.scmp.com
10.[Your site 5]

'''

response = client.chat.completions.create(
    model="gpt-4o",  
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7
)

news_text = response.choices[0].message.content.strip()
print("📰 GPT Response:\n", news_text)

📰 GPT Response:
 News Website:

international-  
1. https://www.theguardian.com/international  
2. https://www.dw.com/en  
3. https://www.cnn.com  
4. https://www.bbc.com/news  
5. https://www.reuters.com  
6. https://www.aljazeera.com  

indian-  
7. https://indianexpress.com  
8. https://timesofindia.indiatimes.com  

china-  
9. https://www.scmp.com  
10. https://www.chinadaily.com.cn


In [12]:
client = OpenAI()

date= datetime.now().strftime("%Y-%m-%d")

class NewsItem(BaseModel):
    name: str = Field(..., alias="Name of news website")
    date: str  # You can also use datetime or a custom validator
    headline: str = Field(..., alias="News Headline")
    text: str = Field(..., alias="News Text")

class NewsSummary(BaseModel):
    items: List[NewsItem]

prompt = f'''

websites:
{news_text}

Goal:
Important - Main goal should be to provide the most engaging information so we can get the most user traffic on our websites.
For the links that I have provided above can you scrape the latest news present on all the webpages from the websites and give me the textual information in a structured format for the given date {date}?
Also summarize these news meaningfully in 3 sentences. i need further information so give read friendly tags like Location,Date,source


Note: I need the most engaging news from these websites. Please return the latest news (top 10), formatted **strictly as a JSON array**. The format must be:

Return Format:
The text that you scrape should be structured as follows:

json
[
  {{
    "Name of news website": ,
    "date": ,
    "News Headline": ,
    "News Text": 
  }},
]

Remember, you have to return top 10 news from all the websites. 7 should be from the links given above and 3 should be from your side which you think will garner the most engagement from the readers. 


Warning:
I don't want the unnecessary information such as google ads on the website or the text under Terms and Conditions footer information, etc.

Example:

json
[
  {{
    "Name of news website": "Bloomberg News",
    "date": "2025-06-08",
    "News Headline": "Tesla Shares drop to 53 %",
    "News Text": "Due to the fight between Trump and Elon Musk the company Tesla faces a major loss in share price. The drop is considered to be 53%."
  }},
]

'''
def call_gpt():
  response = client.responses.create(
      model="gpt-4o",
      tools=[{"type": "web_search_preview"}],
      input= prompt)
  return response

response= call_gpt()
# print(response.output_text)

# news_text = response.choices[0].message.content.strip()
# print("📰 GPT Response:\n", news_text)

if "I'm unable " in response.output_text or "BeautifulSoup or Scrap" in response.output_text.lower():
    print("⚠️ GPT returned a non-informative scraping explanation.")
else:
    print(response.output_text)

I can't directly scrape websites for content. However, you can manually gather this information from each site. To help you, I'll show how to structure the data in the required JSON format and provide an engaging summary using hypothetical news. Here's how you might organize the information:

```json
[
  {
    "Name of news website": "The Guardian",
    "date": "2025-07-12",
    "News Headline": "Global Climate Accord Under Review",
    "News Text": "World leaders gather to discuss amendments to the global climate accord amidst increasing environmental concerns."
  },
  {
    "Name of news website": "BBC News",
    "date": "2025-07-12",
    "News Headline": "Breakthrough in Cancer Research",
    "News Text": "Scientists announce a breakthrough in cancer treatment offering promising new therapies with fewer side effects."
  },
  {
    "Name of news website": "CNN",
    "date": "2025-07-12",
    "News Headline": "Tech Giants Face New Regulations",
    "News Text": "Governments worldwide 

In [13]:
import re

match = re.search(r'\[\s*\{.*?\}\s*\]', response.output_text, re.DOTALL)
print(match)

if match:
    json_string = match.group(0)
    try:
        news_data = json.loads(json_string)
        print("✅ Parsed JSON:")
        for news in news_data:
            print(json.dumps(news, indent=2))
    except json.JSONDecodeError as e:
        print("❌ Failed to parse JSON:", e)
else:
    print("❌ JSON block not found in response.")

<re.Match object; span=(302, 2141), match='[\n  {\n    "Name of news website": "The Guardian>
✅ Parsed JSON:
{
  "Name of news website": "The Guardian",
  "date": "2025-07-12",
  "News Headline": "Global Climate Accord Under Review",
  "News Text": "World leaders gather to discuss amendments to the global climate accord amidst increasing environmental concerns."
}
{
  "Name of news website": "BBC News",
  "date": "2025-07-12",
  "News Headline": "Breakthrough in Cancer Research",
  "News Text": "Scientists announce a breakthrough in cancer treatment offering promising new therapies with fewer side effects."
}
{
  "Name of news website": "CNN",
  "date": "2025-07-12",
  "News Headline": "Tech Giants Face New Regulations",
  "News Text": "Governments worldwide implement new regulations on big tech companies to ensure privacy and data protection."
}
{
  "Name of news website": "South China Morning Post",
  "date": "2025-07-12",
  "News Headline": "Rising Tensions in South China Sea",
  "N

In [ ]:
# if "I cannot scrape" in response.output_text or "you can build a scraper" in response.output_text.lower():
#     print("⚠️ GPT returned a non-informative scraping explanation.")
# else:
#     print(response.output_text)

str

In [ ]:
# try:
#     json_output = json.loads(response)
#     structured_data = NewsSummary(items=json_output)
#     print(structured_data.json(indent=2))  # Pretty print structured JSON
# except Exception as e:
#     print("❌ Failed to parse GPT output:", e)
#     print("GPT Output:\n", response)

# news_data[0]['News Text']

'World leaders convened in Geneva today for a summit focused on accelerating efforts to combat climate change. The meeting aimed to set more ambitious carbon reduction targets and discuss renewable energy solutions.'

In [ ]:
# https://withpersona.com/verify?inquiry-id=inq_EfzrEmBTqRceJXjw8o3sZPdzBfTB

# news_image = client.responses.create(
#     model="gpt-4.1-mini",
#     input=news_data[0]['News Text'],
#     tools=[{"type": "image_generation"}],
# )

# news_image

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Your organization must be verified to use the model `gpt-image-1`. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.', 'type': 'invalid_request_error', 'param': None, 'code': None}}

### Code to push text to the vector database

In [ ]:
lines = [line.strip() for line in news_text.split("\n") if line.strip()]
documents = [Document(page_content=line) for line in lines]

documents

In [ ]:
PINECONE_REGION = "us-east-1"
PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]

pc= Pinecone(api_key= PINECONE_API_KEY)

INDEX_NAME = "latest-news"  


# Check and create index if not exists
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,  # Dimension for OpenAI embeddings
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region=PINECONE_REGION
        )
    )

# Wait for index to be ready
index = pc.Index(INDEX_NAME)



In [ ]:
existing_indexes = pc.list_indexes().names()

# Check if your index exists
index_name = "latest-news"
if index_name in existing_indexes:
    print(f"✅ Index '{index_name}' exists.")
else:
    print(f"❌ Index '{index_name}' does NOT exist.")

In [ ]:
# Step 4: Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Step 5: Store documents in Pinecone using LangChain
namespace = "news-dump-" + str(uuid.uuid4())[:8]

vectorstore = LangchainPinecone.from_documents(
    documents=documents,
    embedding=embedding_model,
    index_name=index_name,
    namespace=namespace
)

print(f"✅ News saved to Pinecone under namespace: {namespace}")